# Stage 2 (step-by-step): ASR + alignment + stability + rate + gate

Uses **`StreamingAdapter`** with **`use_rate_controller=True`**, **`EarlyCommitGate`**, and the same Whisper/window helpers.
Full ASR loss needs **`AutoModelForCausalLM`** forward with `inputs_embeds` + `labels` (see stage2 trainer). Below we **verify each loss term** is wired; then optional combined step.


In [ ]:
import os
import sys
import torch.nn as nn
import torch.nn.functional as F

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "info")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

_nb = os.getcwd()
if os.path.basename(_nb) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(_nb, ".."))
else:
    PROJECT_ROOT = os.environ.get(
        "AUDIO_STREAM_ADAPTER_ROOT",
        os.path.abspath(os.path.join(_nb, "..")),
    )

SRC_ROOT = os.path.join(PROJECT_ROOT, "src")
TRAINING_DIR = os.path.join(PROJECT_ROOT, "training")

if not os.path.isdir(os.path.join(SRC_ROOT, "adapter")):
    raise FileNotFoundError(f"Expected package at {SRC_ROOT}/adapter — open notebook from notebooks/ or set AUDIO_STREAM_ADAPTER_ROOT")

for _p in (SRC_ROOT, PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

os.chdir(TRAINING_DIR)
os.makedirs(os.path.join(PROJECT_ROOT, "checkpoints"), exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAINING_DIR:", TRAINING_DIR)
print("cwd:", os.getcwd())


PROJECT_ROOT: /home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter
TRAINING_DIR: /home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter/training
cwd: /home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter/training
Imports OK — QFormerLayer module: adapter.cross_attention


In [ ]:
# Only these adapter modules (no stage*_trainer imports)

from adapter_llm_pipeline import (
    WhisperAdapterLLMPipeline,
    whisper_waveform_to_encoder_windows,
    windows_tensor_to_batch_list,
)
from adapter.cross_attention import QFormerLayer
from adapter.early_commit_gate import EarlyCommitGate
from adapter.rate_controller import AdaptiveRateController
from adapter.stability_buffer import StabilityBuffer
from adapter.streaming_adapter import StreamingAdapter
from adapter.windowing import WhisperFrameWindowizer

print("Imports OK — QFormerLayer module:", QFormerLayer.__module__)

## Step 1 — `AdaptiveRateController` path inside `forward_window`


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

adapter = StreamingAdapter(
    d_encoder=768, d_llm=4096, num_queries=4, num_layers=2, num_heads=4,
    d_ffn=2048, dropout=0.0, use_rate_controller=True, target_rate=2.0,
    cross_layer_in_between=1,
).to(DEVICE, dtype=DTYPE)
adapter.train()

win = torch.randn(1, 40, 768, device=DEVICE, dtype=DTYPE)
adapter.reset_streaming_state()
out = adapter.forward_window(win)
assert out["sparse_loss"] is not None and out["rate_loss"] is not None
print("sparse", float(out["sparse_loss"]), "rate", float(out["rate_loss"]), "gates", out["gate_scores"].shape)


## Step 2 — `EarlyCommitGate` on accumulated embeddings


In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

gate = EarlyCommitGate(d_llm=4096, hidden_dim=256, threshold=0.5, latency_weight=0.1).to(DEVICE, dtype=DTYPE)
gate.train()

# two fake windows of tokens
tok_a = torch.randn(1, 4, 4096, device=DEVICE, dtype=DTYPE)
tok_b = torch.randn(1, 4, 4096, device=DEVICE, dtype=DTYPE)
acc = torch.cat([tok_a, tok_b], dim=1)
gr = gate(acc, timestep=2, total_timesteps=5)
print(gr.keys(), "gate_loss", float(gr["gate_loss"]))


## Step 3 — Combined scalar (structure of stage 2 total loss)
Tune weights to match `adapter_asr_trainer.py`. `L_asr` omitted here — plug in `llm_model(..., inputs_embeds=..., labels=...).loss` when Qwen is loaded.


In [ ]:
import torch
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

LAMBDA_ALIGN = 0.1
LAMBDA_STABILITY = 0.05
LAMBDA_SPARSE = 0.01
LAMBDA_RATE = 0.001
LAMBDA_GATE = 0.1

def contrastive_loss(audio_tokens, text_embeddings, temperature=0.07):
    b = audio_tokens.shape[0]
    a = F.normalize(audio_tokens.float().mean(dim=1), dim=-1)
    t = F.normalize(text_embeddings.float().mean(dim=1), dim=-1)
    return F.cross_entropy((a @ t.T) / temperature, torch.arange(b, device=a.device))

# fake tensors
audio = torch.randn(2, 8, 4096, device=DEVICE, dtype=DTYPE, requires_grad=True)
text = torch.randn(2, 12, 4096, device=DEVICE, dtype=DTYPE)

L_asr = torch.tensor(0.0, device=DEVICE, requires_grad=True)  # replace with real CE
L_align = contrastive_loss(audio, text)
L_stab = torch.tensor(0.01, device=DEVICE, dtype=DTYPE)
L_sparse = torch.tensor(0.02, device=DEVICE, dtype=DTYPE)
L_rate = torch.tensor(0.03, device=DEVICE, dtype=DTYPE)
L_gate = torch.tensor(0.04, device=DEVICE, dtype=DTYPE)

L = L_asr + LAMBDA_ALIGN * L_align + LAMBDA_STABILITY * L_stab + LAMBDA_SPARSE * L_sparse + LAMBDA_RATE * L_rate + LAMBDA_GATE * L_gate
L.backward()
print("L_total", float(L), "audio grad", audio.grad.abs().mean().item())


## Final: mini training loop + checkpoint (sanity)

This runs a short joint update of **adapter + gate** with:
- `L_align` (contrastive)
- `L_stability` (from adapter)
- `L_sparse` / `L_rate` (from rate controller)
- `L_gate` (from `EarlyCommitGate`)

`L_asr` is set to 0 by default to avoid loading the full LLM in a notebook sanity run. Toggle `RUN_ASR=True` if you want to include the real LM loss.


In [ ]:
import os
import time
import torch
import torch.nn.functional as F

SAVE_PATH = os.path.join("checkpoints", "adapter_notebook_adapter_gate.pt")
STEPS = 3
WINDOW_CAP = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

from transformers import WhisperForConditionalGeneration, WhisperProcessor, AutoTokenizer, AutoModel

WHISPER_ID = "openai/whisper-small"
LLM_ID = "Qwen/Qwen3-8B"

whisper_processor = WhisperProcessor.from_pretrained(WHISPER_ID)
whisper_full = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True,
).to(DEVICE)
whisper_full.eval()
for p in whisper_full.parameters():
    p.requires_grad = False

llm_tok = AutoTokenizer.from_pretrained(LLM_ID)
llm_tok.pad_token = llm_tok.eos_token
emb_model = AutoModel.from_pretrained(LLM_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True).to(DEVICE)
emb_model.eval()
for p in emb_model.parameters():
    p.requires_grad = False
text_embedder = emb_model.get_input_embeddings()

windowizer = WhisperFrameWindowizer(window_seconds=0.8, stride_seconds=0.4)
adapter = StreamingAdapter(
    d_encoder=768, d_llm=4096, num_queries=4, num_layers=2, num_heads=4,
    d_ffn=2048, dropout=0.0, use_rate_controller=True, target_rate=2.0, cross_layer_in_between=1,
).to(DEVICE, dtype=DTYPE)
adapter.train()

gate = EarlyCommitGate(d_llm=4096, hidden_dim=256, threshold=0.5, latency_weight=0.1).to(DEVICE, dtype=DTYPE)
gate.train()

params = list(adapter.parameters()) + list(gate.parameters())
opt = torch.optim.AdamW(params, lr=5e-5)

LAMBDA_ALIGN = 0.1
LAMBDA_STABILITY = 0.05
LAMBDA_SPARSE = 0.01
LAMBDA_RATE = 0.001
LAMBDA_GATE = 0.1
TEMPERATURE = 0.07


def contrastive_loss(audio_tokens, text_embeddings, temperature=0.07):
    b = audio_tokens.shape[0]
    a = F.normalize(audio_tokens.float().mean(dim=1), dim=-1)
    t = F.normalize(text_embeddings.float().mean(dim=1), dim=-1)
    return F.cross_entropy((a @ t.T) / temperature, torch.arange(b, device=a.device))


def pad_tokens(batch_tokens: list[torch.Tensor]) -> torch.Tensor:
    max_len = max(t.shape[1] for t in batch_tokens)
    out = []
    for t in batch_tokens:
        if t.shape[1] < max_len:
            pad = torch.zeros(1, max_len - t.shape[1], t.shape[2], device=t.device, dtype=t.dtype)
            t = torch.cat([t, pad], dim=1)
        out.append(t)
    return torch.cat(out, dim=0)

print("Running", STEPS, "steps... saving to", SAVE_PATH)
losses = []
t0 = time.time()
for step in range(STEPS):
    texts = [f"gt transcript {step} a", f"gt transcript {step} b"]
    tt = llm_tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        gt_embeds = text_embedder(tt["input_ids"]).float()

    utt_tokens = []
    total_stab = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_sparse = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_rate = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_gate = torch.zeros((), device=DEVICE, dtype=torch.float32)

    for _ in range(2):
        wave = torch.randn(19200, device=DEVICE, dtype=torch.float32)
        enc, windows = whisper_waveform_to_encoder_windows(
            wave,
            whisper_processor=whisper_processor,
            whisper_model=whisper_full,
            windowizer=windowizer,
            device=DEVICE,
            torch_dtype=DTYPE,
        )
        win_list = windows_tensor_to_batch_list(windows)
        adapter.reset_streaming_state()
        chunks = []
        for t, w in enumerate(win_list[:WINDOW_CAP]):
            o = adapter.forward_window(w.to(DEVICE, dtype=DTYPE))
            chunks.append(o["tokens"])
            total_stab = total_stab + o["stability_loss"].float()
            if o["sparse_loss"] is not None:
                total_sparse = total_sparse + o["sparse_loss"].float()
            if o["rate_loss"] is not None:
                total_rate = total_rate + o["rate_loss"].float()
            if t > 0:
                acc = torch.cat(chunks[:t], dim=1)
                gr = gate(acc, timestep=t, total_timesteps=WINDOW_CAP)
                total_gate = total_gate + gr["gate_loss"].float()
        utt_tokens.append(torch.cat(chunks, dim=1))

    audio_tokens = pad_tokens(utt_tokens).float()
    L_asr = torch.zeros((), device=DEVICE)  # set non-zero if you wire LLM CE
    L_align = contrastive_loss(audio_tokens, gt_embeds, temperature=TEMPERATURE)
    L_stab = total_stab / 2.0
    L_sparse = total_sparse / 2.0
    L_rate = total_rate / 2.0
    L_gate = total_gate / 2.0

    L = L_asr + LAMBDA_ALIGN * L_align + LAMBDA_STABILITY * L_stab + LAMBDA_SPARSE * L_sparse + LAMBDA_RATE * L_rate + LAMBDA_GATE * L_gate

    opt.zero_grad()
    L.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt.step()

    losses.append(float(L.detach().cpu()))
    print(f"step {step}: L={losses[-1]:.4f} align={float(L_align):.4f} stab={float(L_stab):.4f} sparse={float(L_sparse):.4f} rate={float(L_rate):.4f} gate={float(L_gate):.4f}")

ckpt = {
    "stage": 2,
    "adapter_state_dict": adapter.state_dict(),
    "gate_state_dict": gate.state_dict(),
    "avg_loss": sum(losses) / len(losses),
}
torch.save(ckpt, SAVE_PATH)
print("Saved checkpoint.")
print("Elapsed_s:", time.time() - t0)


## Full training (script-equivalent)


In [ ]:
# Full Stage 2 training (notebook)
# This follows adapter_asr_trainer.py structure, but keeps everything visible.
# Saves `checkpoints/adapter_adapter.pt`.

import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, WhisperForConditionalGeneration, WhisperProcessor

from training.utils.common import (
    AdapterCheckpoint,
    LibriSpeechPairs,
    default_librispeech_root_from_training_dir,
    load_mono_waveform_16k,
    save_checkpoint,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

EPOCHS = 10
BATCH_SIZE = 2
LR = 5e-5
LAMBDA_ALIGN = 0.1
LAMBDA_STABILITY = 0.05
LAMBDA_SPARSE = 0.01
LAMBDA_RATE = 0.001
TEMPERATURE = 0.07
SAVE_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "adapter_adapter.pt")
MAX_WINDOWS_PER_UTT = None

WHISPER_ID = "openai/whisper-small"
LLM_ID = "Qwen/Qwen3-8B"
DATASET_ROOT = default_librispeech_root_from_training_dir(TRAINING_DIR)

print("DEVICE", DEVICE, "DTYPE", DTYPE)
print("DATASET_ROOT", DATASET_ROOT)

whisper_processor = WhisperProcessor.from_pretrained(WHISPER_ID)
whisper_full = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True,
).to(DEVICE)
whisper_full.eval()
for p in whisper_full.parameters():
    p.requires_grad = False

windowizer = WhisperFrameWindowizer(window_seconds=0.8, stride_seconds=0.4)

llm_tok = AutoTokenizer.from_pretrained(LLM_ID)
llm_tok.pad_token = llm_tok.eos_token
if DEVICE == "cuda":
    torch.cuda.init()
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True, device_map="auto")
llm.eval()
for p in llm.parameters():
    p.requires_grad = False
text_embedder = llm.get_input_embeddings()

adapter = StreamingAdapter(
    d_encoder=768, d_llm=4096, num_queries=4, num_layers=2, num_heads=4,
    d_ffn=2048, dropout=0.1, ema_alpha=0.8, learnable_ema=False,
    use_rate_controller=True, target_rate=2.0, cross_layer_in_between=1,
).to(DEVICE, dtype=DTYPE)
adapter.train()

gate = EarlyCommitGate(d_llm=4096, hidden_dim=256, threshold=0.5, latency_weight=0.1).to(DEVICE, dtype=DTYPE)
gate.train()

params = list(adapter.parameters()) + list(gate.parameters())
opt = torch.optim.AdamW(params, lr=LR)


def contrastive_loss(audio_tokens, text_embeddings, temperature=0.07):
    b = audio_tokens.shape[0]
    a = F.normalize(audio_tokens.float().mean(dim=1), dim=-1)
    t = F.normalize(text_embeddings.float().mean(dim=1), dim=-1)
    return F.cross_entropy((a @ t.T) / temperature, torch.arange(b, device=a.device))


def pad_tokens(batch_tokens: list[torch.Tensor]) -> torch.Tensor:
    max_len = max(t.shape[1] for t in batch_tokens)
    out = []
    for t in batch_tokens:
        if t.shape[1] < max_len:
            pad = torch.zeros(1, max_len - t.shape[1], t.shape[2], device=t.device, dtype=t.dtype)
            t = torch.cat([t, pad], dim=1)
        out.append(t)
    return torch.cat(out, dim=0)


dataset = LibriSpeechPairs(DATASET_ROOT)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

for epoch in range(EPOCHS):
    total = 0.0
    for step, batch in enumerate(loader):
        audio_paths, texts = batch

        gt = llm_tok(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
        gt_ids = gt["input_ids"]
        gt_mask = gt["attention_mask"]

        with torch.no_grad():
            gt_embeds = text_embedder(gt_ids)

        utt_tokens = []
        stab_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)
        sparse_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)
        rate_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)
        gate_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)

        for p in audio_paths:
            wave = load_mono_waveform_16k(p)
            enc, windows = whisper_waveform_to_encoder_windows(
                wave,
                whisper_processor=whisper_processor,
                whisper_model=whisper_full,
                windowizer=windowizer,
                device=DEVICE,
                torch_dtype=DTYPE,
            )
            win_list = windows_tensor_to_batch_list(windows)
            if MAX_WINDOWS_PER_UTT is not None:
                win_list = win_list[:MAX_WINDOWS_PER_UTT]

            adapter.reset_streaming_state()
            chunks = []
            for t, w in enumerate(win_list):
                o = adapter.forward_window(w.to(DEVICE, dtype=DTYPE))
                chunks.append(o["tokens"])
                stab_sum = stab_sum + o["stability_loss"].float()
                if o["sparse_loss"] is not None:
                    sparse_sum = sparse_sum + o["sparse_loss"].float()
                if o["rate_loss"] is not None:
                    rate_sum = rate_sum + o["rate_loss"].float()
                if t > 0:
                    acc = torch.cat(chunks[:t], dim=1)
                    gr = gate(acc, timestep=t, total_timesteps=len(win_list))
                    gate_sum = gate_sum + gr["gate_loss"].float()
            utt_tokens.append(torch.cat(chunks, dim=1))

        audio_tokens = pad_tokens(utt_tokens)

        # ASR LM loss (teacher forcing)
        bos_id = llm_tok.bos_token_id if llm_tok.bos_token_id is not None else llm_tok.eos_token_id
        bos_embed = text_embedder(torch.tensor([[bos_id]], device=DEVICE).expand(audio_tokens.shape[0], -1))
        inputs_embeds = torch.cat([audio_tokens, bos_embed], dim=1)
        pre_labels = torch.full((audio_tokens.shape[0], audio_tokens.shape[1] + 1), -100, dtype=torch.long, device=DEVICE)
        gt_shift = gt_ids[:, 1:]
        labels = torch.cat([pre_labels, gt_shift], dim=1)
        inputs_embeds = torch.cat([inputs_embeds, text_embedder(gt_shift)], dim=1)
        pre_mask = torch.ones((audio_tokens.shape[0], audio_tokens.shape[1] + 1), device=DEVICE)
        attn = torch.cat([pre_mask, gt_mask[:, 1:]], dim=1)

        asr_out = llm(inputs_embeds=inputs_embeds, labels=labels, attention_mask=attn)
        L_asr = asr_out.loss

        L_align = contrastive_loss(audio_tokens.float(), gt_embeds.float(), temperature=TEMPERATURE)
        denom = float(len(audio_paths))
        L_stab = stab_sum / denom
        L_sparse = sparse_sum / denom
        L_rate = rate_sum / denom

        L = L_asr + LAMBDA_ALIGN * L_align + LAMBDA_STABILITY * L_stab + LAMBDA_SPARSE * L_sparse + LAMBDA_RATE * L_rate + 0.1 * gate_sum

        opt.zero_grad()
        L.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()

        total += float(L.detach().cpu())
        if step % 10 == 0:
            print(f"epoch {epoch+1}/{EPOCHS} step {step}: loss={float(L):.4f} asr={float(L_asr):.4f} align={float(L_align):.4f}")

        if step == 0 and epoch == 0:
            print("(notebook note) remove this early break for full training")
            break

    avg = total / max(1, (step + 1))
    save_checkpoint(SAVE_PATH, AdapterCheckpoint(stage=2, adapter_state_dict=adapter.state_dict(), gate_state_dict=gate.state_dict(), avg_loss=avg))
    print("Saved", SAVE_PATH, "avg_loss", avg)

print("Stage 2 complete")
